# Biggest Polities: Sampling Classification

Classify randomly sampled people into polities using the deterministic assignment table.
This handles cases where modern country borders map cleanly onto historical polities.

For people who can't be deterministically assigned (indeterminate), we'll eventually
use LLM calls to identify their polity based on birth location and year.

**Pipeline:**
1. Load 1M sample of randomly sampled people (weighted by historical births)
2. Apply `assign_polity(country, year)` to each Holocene person
3. Analyze: what fraction is assigned vs indeterminate vs pre-polity?
4. Identify the biggest gaps (where LLM calls would add the most value)

In [1]:
import os
import sys
from tqdm import tqdm

import sys
import os
sys.path.insert(0, '..')
os.chdir('..')

from person import sample_person
from llm_utils import get_client, make_langchain_messages, CostTracker, extract_json

os.chdir('BiggestPolities')

# Ensure we can import from both BiggestPolities/ and generating_code/
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
parent_dir = os.path.dirname(notebook_dir)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

# Change to generating_code/ so data file paths resolve correctly
os.chdir(parent_dir)

import numpy as np
import dill
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

sys.path.insert(0, 'BiggestPolities')
from polity_assignments_data import assign_polity, ASSIGNMENT

In [2]:
N = 10**6
PICKLE_FILE = f'famous_person_sample_{N}.pkl'

import os
if os.path.exists(PICKLE_FILE):
    print(f'Loading {N} people from {PICKLE_FILE}...')
    with open(PICKLE_FILE, 'rb') as f:
        people = dill.load(f)
else:
    print(f'Sampling {N} people...')
    people = [sample_person(light=True) for _ in tqdm(range(N), desc='Sampling')]
    with open(PICKLE_FILE, 'wb') as f:
        dill.dump(people, f)
    print(f'Saved to {PICKLE_FILE}')

Loading 1000000 people from famous_person_sample_1000000.pkl...


## Classify each person

In [3]:
records = []
for p in people:
    if p.era == 'Paleolithic':
        records.append({
            'birth_year': p.birth_year,
            'country': None,
            'subregion': None,
            'era': 'Paleolithic',
            'polity': 'no_known_polities',
            'status': 'no_known_polities',
        })
    else:
        country = p.location.country
        subregion = getattr(p.location, 'subregion', None)
        polity = assign_polity(country, p.birth_year, subregion=subregion)
        status = polity if polity in ('no_known_polities', 'indeterminate', 'not_relevant') else 'assigned'
        records.append({
            'birth_year': p.birth_year,
            'country': country,
            'subregion': subregion,
            'era': 'Holocene',
            'polity': polity,
            'status': status,
        })

df = pd.DataFrame(records)

total = len(df)
for status in ['assigned', 'indeterminate', 'not_relevant', 'no_known_polities']:
    n = (df['status'] == status).sum()
    print(f'  {status:<20} {n:>6,} ({100*n/total:.1f}%)')
print(f'  {"Total":<20} {total:>6,}')

  assigned             379,310 (37.9%)
  indeterminate        170,452 (17.0%)
  not_relevant         237,347 (23.7%)
  no_known_polities    212,891 (21.3%)
  Total                1,000,000


In [4]:
# Build pool of people needing LLM classification (indeterminate + not_relevant)
unclassified_people = [
    (i, p) for i, p in enumerate(people) 
    if records[i]['status'] in ('indeterminate', 'not_relevant')
]
print(f"Unclassified people for LLM: {len(unclassified_people):,}")
print(f"  indeterminate: {sum(1 for r in records if r['status'] == 'indeterminate'):,}")
print(f"  not_relevant:  {sum(1 for r in records if r['status'] == 'not_relevant'):,}")


Unclassified people for LLM: 407,799
  indeterminate: 170,452
  not_relevant:  237,347


## Top polities by sampled births

In [5]:
assigned = df[df['status'] == 'assigned']
polity_counts = assigned['polity'].value_counts()

# Show top 50 with ±1σ error bars
n = len(df)
top50 = polity_counts.head(50)
print(f'{"Rank":<5} {"Polity":<55} {"Count":>6} {"% of all":>14}')
print('-' * 83)
for i, (polity, count) in enumerate(top50.items(), 1):
    p = count / n
    se = np.sqrt(p * (1 - p) / n) * 100
    print(f'{i:<5} {polity:<55} {count:>6} {100*p:>6.3f} ± {se:.3f}%')

print(f'\n... {len(polity_counts)} total polities assigned')

Rank  Polity                                                   Count       % of all
-----------------------------------------------------------------------------------
1     Qing Dynasty                                             37765  3.776 ± 0.019%
2     Republic of India                                        28322  2.832 ± 0.017%
3     People's Republic of China                               24531  2.453 ± 0.015%
4     British Raj                                              19098  1.910 ± 0.014%
5     Ming Dynasty                                             12454  1.245 ± 0.011%
6     Roman Empire                                             11897  1.190 ± 0.011%
7     United States                                             7581  0.758 ± 0.009%
8     Southern Song Dynasty                                     7185  0.719 ± 0.008%
9     Holy Roman Empire                                         7009  0.701 ± 0.008%
10    Russian Empire                                            641

## Gap analysis: where are the biggest indeterminate clusters?

These are the (country, century) pairs with the most unassigned people — i.e., where
adding polity assignments (either deterministically or via LLM) would have the biggest payoff.

In [6]:
indet = df[df['status'] == 'indeterminate'].copy()
indet['century'] = (indet['birth_year'] // 100) * 100

# By country (summing across all centuries)
by_country = indet.groupby('country').size().sort_values(ascending=False)
print(f'{"Country":<25} {"Indeterminate":>13} {"% of all":>8}')
print('-' * 48)
for country, count in by_country.head(20).items():
    print(f'{country:<25} {count:>13,} {100*count/total:>7.2f}%')

print(f'\nTop 20 countries account for {100*by_country.head(20).sum()/total:.1f}% of all people')

Country                   Indeterminate % of all
------------------------------------------------
India                            59,172    5.92%
China                            29,273    2.93%
Bangladesh                        8,401    0.84%
Italy                             8,174    0.82%
Russia                            5,479    0.55%
Pakistan                          4,779    0.48%
France                            4,085    0.41%
Turkey                            3,818    0.38%
Ukraine                           3,591    0.36%
Afghanistan                       3,548    0.35%
Poland                            3,144    0.31%
Germany                           2,114    0.21%
Ethiopia                          2,042    0.20%
Saudi Arabia                      1,852    0.19%
Romania                           1,668    0.17%
Iran                              1,648    0.16%
Morocco                           1,439    0.14%
Algeria                           1,362    0.14%
Brazil              

## LLM classification of indeterminate people

Sample a random subset of indeterminate people, batch them by (country, century),
and ask an LLM to classify each into a polity.

**Batching strategy**: Group by (country, century) so the LLM can load the relevant
historical context once per batch. Cap at ~20 people per batch.

In [7]:
from llm_classification import (
    run_classification, sample_and_batch, classify_batch,
    get_canonical_polities, EXTRA_CANONICAL_POLITIES,
)

print(f"Unclassified people: {len(unclassified_people):,}")


Unclassified people: 407,799


In [8]:
# Load LLM classification results from pickle (instead of re-running)
import dill

LLM_PICKLE = 'BiggestPolities/llm_classification_10000.pkl'
with open(LLM_PICKLE, 'rb') as f:
    save_data = dill.load(f)

results = save_data['results']
sample_records = save_data['sample_records']
raw_responses = save_data['raw_responses']
N_SAMPLE = save_data['n_sample']
MODEL = save_data.get('model', 'gpt-5.2')

In [9]:
## To re-run LLM classification instead, uncomment below and comment out the pickle load:
# N_SAMPLE = 10_000
# MODEL = 'gpt-5.2'
# results, sample_records, raw_responses, tracker = run_classification(
#     unclassified_people, N_SAMPLE, model=MODEL, batch_size=100, seed=42, max_workers=10
# )
# Save results (already saved — uncomment to re-save after a new LLM run)
# import dill
# 
# save_data = {
#     'results': results,
#     'sample_records': sample_records,
#     'raw_responses': raw_responses,
#     'n_sample': len(sample_records),
#     'model': MODEL,
#     'n_unclassified_total': len(unclassified_people),
# }
# fname = f'llm_classification_{len(sample_records)}.pkl'
# with open(fname, 'wb') as f:
#     dill.dump(save_data, f)
# print(f"Saved to {fname}")

In [10]:
# Show results + name consistency check
print(f"{'Year':>8} {'Lat':>7} {'Lon':>7} {'Country':<18} {'Subregion':<22} {'Polity'}")
print('-' * 100)
for orig_idx, country, subregion, year, year_str, lat, lon in sample_records[:10]:
    polity = results.get(orig_idx, '???')
    print(f"{year_str:>8} {lat:>7.1f} {lon:>7.1f} {country:<18} {subregion:<22} {polity}")

    Year     Lat     Lon Country            Subregion              Polity
----------------------------------------------------------------------------------------------------
 1980 BC    35.3    70.2 Afghanistan        Laghmān                NO_POLITY
 1946 BC    35.1    65.9 Afghanistan        Ghowr                  NO_POLITY
 1895 BC    37.2    68.9 Afghanistan        Kondoz [Kunduz]        NO_POLITY
 1875 BC    36.0    68.9 Afghanistan        Baghlān                NO_POLITY
 1820 BC    34.6    70.6 Afghanistan        Nangrahār [Nangarhār]  NO_POLITY
 1797 BC    35.5    69.8 Afghanistan        Kāpīsā                 NO_POLITY
 1577 BC    34.4    69.2 Afghanistan        Kābul [Kābol]          NO_POLITY
 1539 BC    32.6    68.3 Afghanistan        Ghaznī                 NO_POLITY
 1486 BC    35.3    62.0 Afghanistan        Bādghīs                NO_POLITY
 1453 BC    36.1    69.7 Afghanistan        Takhār                 NO_POLITY


In [11]:
# Name consistency
from llm_classification import get_canonical_polities
all_canonical = set()
for rec in sample_records:
    all_canonical.update(get_canonical_polities(rec[1]))

canonical_used = {p for p in results.values() if p in all_canonical}
non_canonical = {p for p in results.values() if p not in all_canonical and p != 'NO_POLITY'}

print(f"\nCanonical: {len(canonical_used)} | New: {len(non_canonical)}")
#for n in sorted(non_canonical):
#    print(f"  • {n}")


Canonical: 228 | New: 882


## Combined ranking: deterministic + LLM sample

Merge the deterministic assignments (375K people) with the LLM-classified sample 
(extrapolated to represent all 187K indeterminate people).

In [12]:
import importlib
import llm_classification
importlib.reload(llm_classification)
from llm_classification import merge_polity_name

# --- Deterministic counts, with alias merging ---
det_raw = df[df['status'] == 'assigned']['polity'].map(merge_polity_name)
det_counts = det_raw.value_counts()

# --- LLM sample counts, with alias merging, scaled up ---
n_unclassified_total = (df['status'].isin(['indeterminate', 'not_relevant'])).sum()
n_sample_classified = len(results)
scale_factor = n_unclassified_total / n_sample_classified

print(f"Deterministic: {det_counts.sum():,} assigned to {len(det_counts)} polities")
print(f"LLM sample: {n_sample_classified} classified, scale factor = {scale_factor:.1f}x")
print(f"  (each sampled person represents ~{scale_factor:.0f} unclassified people)")

llm_merged = Counter(merge_polity_name(p) for p in results.values())
llm_merged.pop('NO_POLITY', None)

llm_counts_raw = dict(llm_merged)
llm_counts_scaled = {polity: count * scale_factor for polity, count in llm_counts_raw.items()}

# --- Merge ---
combined = dict(det_counts)
for polity, scaled_count in llm_counts_scaled.items():
    combined[polity] = combined.get(polity, 0) + scaled_count

# --- Display with combined error bars ---
N_total = len(df)
all_polities = set(det_counts.index) | set(llm_counts_scaled.keys())

rows = []
for polity in all_polities:
    det = det_counts.get(polity, 0)
    llm_scaled = llm_counts_scaled.get(polity, 0)
    llm_raw = llm_counts_raw.get(polity, 0)
    total = det + llm_scaled
    
    # Binomial variance from MC sample + sub-sample scaling variance
    # Using (llm_raw + 1) so polities with 0 LLM hits still get nonzero variance
    var_det = det * (1 - det / N_total)
    var_llm = (llm_raw + 1) * (1 - (llm_raw + 1) / n_sample_classified) * scale_factor**2
    se_total = np.sqrt(var_det + var_llm)
    rows.append((polity, det, llm_scaled, llm_raw, total, se_total))

rows.sort(key=lambda r: -r[4])

print(f"\n{'Rank':<5} {'Polity':<60} {'Det':>7} {'LLM':>7} {'Total':>7} {'% of all':>14}")
print('-' * 100)
for i, (polity, det, llm_scaled, llm_raw, total, se_total) in enumerate(rows, 1):
    pct = 100 * total / N_total
    se_pct = 100 * se_total / N_total
    print(f"{i:<5} {polity:<60} {det:>7,.0f} {llm_scaled:>7,.0f} {total:>7,.0f} {pct:>6.3f} ± {se_pct:.3f}%")

Deterministic: 379,310 assigned to 468 polities
LLM sample: 10000 classified, scale factor = 40.8x
  (each sampled person represents ~41 unclassified people)

Rank  Polity                                                           Det     LLM   Total       % of all
----------------------------------------------------------------------------------------------------
1     Qing Dynasty                                                  37,765     775  38,540  3.854 ± 0.026%
2     Republic of India                                             28,322       0  28,322  2.832 ± 0.017%
3     People's Republic of China                                    24,531     938  25,469  2.547 ± 0.025%
4     British Raj                                                   19,098      82  19,180  1.918 ± 0.015%
5     Ming Dynasty                                                  12,454   1,590  14,044  1.404 ± 0.028%
6     Roman Empire                                                  11,897     979  12,876  1.288 ±

In [13]:
## Grouped ranking: succession groups

importlib.reload(llm_classification)
from llm_classification import POLITY_GROUPS, group_polity_name

# Get the set of group names (not individual polities that pass through)
group_names = set(POLITY_GROUPS.values())

# Apply alias merging first, then grouping
def full_group(name):
    return group_polity_name(merge_polity_name(name))

# --- Deterministic counts, grouped ---
det_grouped = df[df['status'] == 'assigned']['polity'].map(full_group)
det_g_counts = det_grouped.value_counts()

# --- LLM counts, grouped ---
llm_g_merged = Counter(full_group(p) for p in results.values())
llm_g_merged.pop('NO_POLITY', None)

llm_g_raw = dict(llm_g_merged)
llm_g_scaled = {g: count * scale_factor for g, count in llm_g_raw.items()}

# --- Only show the actual groups, not ungrouped pass-throughs ---
g_rows = []
for g in group_names:
    det = det_g_counts.get(g, 0)
    llm_scaled = llm_g_scaled.get(g, 0)
    llm_raw = llm_g_raw.get(g, 0)
    total = det + llm_scaled
    
    var_det = det * (1 - det / N_total)
    var_llm = (llm_raw + 1) * (1 - (llm_raw + 1) / n_sample_classified) * scale_factor**2
    se_total = np.sqrt(var_det + var_llm)
    g_rows.append((g, det, llm_scaled, total, se_total))

g_rows.sort(key=lambda r: -r[3])

print(f"{'Rank':<5} {'Group':<40} {'Det':>9} {'LLM':>9} {'Total':>9} {'% of all':>14}")
print('-' * 90)
for i, (g, det, llm_scaled, total, se_total) in enumerate(g_rows, 1):
    pct = 100 * total / N_total
    se_pct = 100 * se_total / N_total
    print(f"{i:<5} {g:<40} {det:>9,.0f} {llm_scaled:>9,.0f} {total:>9,.0f} {pct:>6.3f} ± {se_pct:.3f}%")

Rank  Group                                          Det       LLM     Total       % of all
------------------------------------------------------------------------------------------
1     Imperial China                              80,432     6,933    87,365  8.736 ± 0.059%
2     Roman civilization                          18,044     3,262    21,306  2.131 ± 0.039%
3     Russian Empire / USSR                       13,589     5,261    18,850  1.885 ± 0.048%
4     Japan (all periods)                         11,462     1,060    12,522  1.252 ± 0.024%
5     United States (broad)                        7,912       612     8,524  0.852 ± 0.019%
6     Ancient Egypt                                1,528     2,528     4,056  0.406 ± 0.033%
7     German Empire (broad)                        2,812       122     2,934  0.293 ± 0.010%
8     Mongol Empire (broad)                          585     1,550     2,135  0.213 ± 0.026%


In [14]:
## Overlay rankings: groups that can overlap with the main ranking

importlib.reload(llm_classification)
from llm_classification import POLITY_OVERLAYS, merge_polity_name

def count_overlay(members, df, results, sample_records, merge_fn, scale_factor):
    """Count people matching a time-aware overlay group.
    
    Returns (total, breakdown_dict) where breakdown_dict maps polity -> count.
    """
    # Build member lookup: polity_name -> list of (start, end) ranges
    member_ranges = {}
    for name, start, end in members:
        member_ranges.setdefault(name, []).append((start, end))
    
    def matches(polity, year):
        ranges = member_ranges.get(polity)
        if not ranges:
            return False
        for start, end in ranges:
            if (start is None or year >= start) and (end is None or year <= end):
                return True
        return False
    
    # Deterministic
    det_breakdown = Counter()
    assigned = df[df['status'] == 'assigned']
    for _, row in assigned.iterrows():
        polity = merge_fn(row['polity'])
        if matches(polity, row['birth_year']):
            det_breakdown[polity] += 1
    
    # LLM sample
    idx_to_year = {rec[0]: rec[3] for rec in sample_records}
    llm_breakdown = Counter()
    for orig_idx, polity_name in results.items():
        polity = merge_fn(polity_name)
        year = idx_to_year.get(int(orig_idx))
        if year is not None and matches(polity, year):
            llm_breakdown[polity] += 1
    
    # Combine
    all_polities = set(det_breakdown) | set(llm_breakdown)
    breakdown = {}
    for p in all_polities:
        breakdown[p] = det_breakdown[p] + llm_breakdown[p] * scale_factor
    
    total = sum(breakdown.values())
    return total, breakdown

# Compute all overlays
print(f"{'Overlay group':<35} {'Total':>9} {'% of all':>10}")
print('-' * 58)

overlay_results = {}
for group_name, members in sorted(POLITY_OVERLAYS.items()):
    total, breakdown = count_overlay(
        members, df, results, sample_records, merge_polity_name, scale_factor
    )
    overlay_results[group_name] = (total, breakdown)
    pct = 100 * total / N_total
    print(f"{group_name:<35} {total:>9,.0f} {pct:>9.3f}%")

# Detailed breakdown
for group_name in sorted(overlay_results):
    total, breakdown = overlay_results[group_name]
    rows = sorted(breakdown.items(), key=lambda x: -x[1])
    print(f"\n{'='*70}")
    print(f"{group_name}: {total:,.0f} ({100*total/N_total:.3f}%)")
    print(f"{'='*70}")
    for m, v in rows:
        print(f"  {m:<55} {v:>9,.0f}")

Overlay group                           Total   % of all
----------------------------------------------------------
British Empire                         39,552     3.955%
Caliphate (mainline)                    4,073     0.407%
Dutch Empire                            3,866     0.387%
French Empire                          10,192     1.019%
Habsburg domains                        7,582     0.758%
Mongol Empire (Genghisid)               7,659     0.766%
Portuguese Empire                       2,348     0.235%
Spanish Empire                          9,047     0.905%

British Empire: 39,552 (3.955%)
  British Raj                                                19,180
  East India Company                                          7,529
  United Kingdom                                              3,167
  Kingdom of England                                          1,484
  Colony and Protectorate of Nigeria                            985
  Dominion of India                                    

## Person-years analysis

The rankings above count **births** — each sampled person counts as 1. But which polities
had the most **person-years** (= integral of population over time)?

Each person is sampled proportional to births in their country-year. Since
$\text{births} = \text{population} \times \text{CBR}/1000$,
re-weighting each person by $1/\text{CBR}$ converts from births-weighted to population-weighted
(i.e. person-years).

We use country-specific CBR from `processed_p1600_data.pkl` (covering 1600–2025).
For pre-1600 Holocene and Paleolithic people we use a default CBR of 42 per 1000.

In [15]:
# Load country-specific CBR data
import dill as dill2
with open('Processed_Data/processed_p1600_data.pkl', 'rb') as f:
    country_data = dill2.load(f)

DEFAULT_CBR = 42.0  # births per 1000 population, used for pre-1600 and Paleolithic

def get_cbr(country, year):
    """Get crude birth rate (per 1000) for a country-year pair."""
    if country is None or country not in country_data:
        return DEFAULT_CBR
    cd = country_data[country]
    if year < cd.years[0] or year > cd.years[-1]:
        return DEFAULT_CBR
    return float(cd.CBR_f(year))

# Compute weights: 1/CBR for each person
cbr_values = np.array([get_cbr(df.loc[i, 'country'], df.loc[i, 'birth_year']) for i in range(len(df))])
weights = 1.0 / cbr_values

# Normalize so weights sum to 1
weights_norm = weights / weights.sum()

# Stats
n_default = np.sum(cbr_values == DEFAULT_CBR)
print(f"People using default CBR: {n_default:,} ({100*n_default/len(df):.1f}%)")
print(f"CBR range: [{cbr_values.min():.1f}, {cbr_values.max():.1f}] per 1000")
print(f"Weight range: [{weights.min():.4f}, {weights.max():.4f}]")
print(f"Effective sample size: {1/np.sum(weights_norm**2):,.0f} (of {len(people):,})")

People using default CBR: 639,352 (63.9%)
CBR range: [4.7, 59.4] per 1000
Weight range: [0.0168, 0.2106]
Effective sample size: 861,531 (of 1,000,000)


In [16]:
# Add weights to the dataframe
df['weight'] = weights
df['weight_norm'] = weights_norm

# --- Deterministic: person-years by polity ---
assigned_w = df[df['status'] == 'assigned'].copy()
assigned_w['polity_merged'] = assigned_w['polity'].map(merge_polity_name)
det_py = assigned_w.groupby('polity_merged')['weight_norm'].sum()

# --- LLM sample: person-years scaled up ---
# For LLM-classified people, we need their weights too
llm_weight_sum = {}
llm_weight_raw_count = {}
for orig_idx, polity_name in results.items():
    merged = merge_polity_name(polity_name)
    if merged == 'NO_POLITY':
        continue
    w = weights_norm[int(orig_idx)]
    llm_weight_sum[merged] = llm_weight_sum.get(merged, 0) + w

# Scale LLM weights: the LLM sample covers n_sample_classified out of n_unclassified_total people
# We need to scale up the weights from the subsample to represent all unclassified people
# The weights are already normalized over all 1M people, so we just scale by the subsampling ratio
unclassified_mask = df['status'].isin(['indeterminate', 'not_relevant'])
total_unclassified_weight = df.loc[unclassified_mask, 'weight_norm'].sum()
sampled_indices = set(int(rec[0]) for rec in sample_records)
sampled_weight = df.loc[list(sampled_indices), 'weight_norm'].sum()
llm_weight_scale = total_unclassified_weight / sampled_weight

print(f"Total weight of unclassified: {total_unclassified_weight:.4f}")
print(f"Weight of LLM subsample: {sampled_weight:.4f}")
print(f"LLM weight scale factor: {llm_weight_scale:.2f}")

llm_py_scaled = {polity: w * llm_weight_scale for polity, w in llm_weight_sum.items()}

# --- Merge deterministic + LLM ---
combined_py = dict(det_py)
for polity, w in llm_py_scaled.items():
    combined_py[polity] = combined_py.get(polity, 0) + w

# --- Display top 50 ---
py_rows = sorted(combined_py.items(), key=lambda x: -x[1])

print(f"\n{'Rank':<5} {'Polity':<55} {'% person-years':>14}")
print('-' * 77)
for i, (polity, w) in enumerate(py_rows[:50], 1):
    print(f"{i:<5} {polity:<55} {100*w:>10.3f}%")

Total weight of unclassified: 0.3668
Weight of LLM subsample: 0.0090
LLM weight scale factor: 40.79

Rank  Polity                                                  % person-years
-----------------------------------------------------------------------------
1     People's Republic of China                                   4.681%
2     Republic of India                                            3.805%
3     Qing Dynasty                                                 3.715%
4     British Raj                                                  1.608%
5     United States                                                1.542%
6     Ming Dynasty                                                 1.260%
7     Roman Empire                                                 1.153%
8     Mughal Empire                                                1.109%
9     Soviet Union                                                 0.925%
10    Republic of Indonesia                                        0.786%
11  

In [17]:
# --- Comparison: births vs person-years ranking ---

# Reconstruct births ranking from combined (built in the earlier combined ranking cell)
births_rank = {p: i+1 for i, (p, _) in enumerate(sorted(combined.items(), key=lambda x: -x[1]))}

# Top 30 comparison table
print(f"{'Rank':<5} {'Polity (by person-years)':<45} {'% PY':>8} {'Births rank':>12} {'% births':>9}")
print('-' * 82)
for i, (polity, w) in enumerate(py_rows[:30], 1):
    br = births_rank.get(polity, '—')
    bp = 100 * combined.get(polity, 0) / N_total
    print(f"{i:<5} {polity:<45} {100*w:>7.3f}% {br:>10} {bp:>8.3f}%")

Rank  Polity (by person-years)                          % PY  Births rank  % births
----------------------------------------------------------------------------------
1     People's Republic of China                      4.681%          3    2.547%
2     Republic of India                               3.805%          2    2.832%
3     Qing Dynasty                                    3.715%          1    3.854%
4     British Raj                                     1.608%          4    1.918%
5     United States                                   1.542%         10    0.811%
6     Ming Dynasty                                    1.260%          5    1.404%
7     Roman Empire                                    1.153%          6    1.288%
8     Mughal Empire                                   1.109%          7    1.245%
9     Soviet Union                                    0.925%         20    0.637%
10    Republic of Indonesia                           0.786%         25    0.553%
11    Russian

In [18]:
# --- Grouped ranking by person-years ---

# Apply full grouping (merge + group) to person-years
def full_group(name):
    return group_polity_name(merge_polity_name(name))

# Deterministic grouped
assigned_w['polity_grouped'] = assigned_w['polity'].map(full_group)
det_gpy = assigned_w.groupby('polity_grouped')['weight_norm'].sum()

# LLM grouped
llm_gw_sum = {}
for orig_idx, polity_name in results.items():
    grouped = full_group(polity_name)
    if grouped == 'NO_POLITY':
        continue
    w = weights_norm[int(orig_idx)]
    llm_gw_sum[grouped] = llm_gw_sum.get(grouped, 0) + w

llm_gpy_scaled = {g: w * llm_weight_scale for g, w in llm_gw_sum.items()}

# Merge - only show actual groups
combined_gpy = {}
for g in group_names:
    det = det_gpy.get(g, 0)
    llm = llm_gpy_scaled.get(g, 0)
    combined_gpy[g] = det + llm

gpy_rows = sorted(combined_gpy.items(), key=lambda x: -x[1])

print(f"{'Rank':<5} {'Group':<40} {'% person-years':>14}")
print('-' * 62)
for i, (g, w) in enumerate(gpy_rows, 1):
    print(f"{i:<5} {g:<40} {100*w:>10.3f}%")

Rank  Group                                    % person-years
--------------------------------------------------------------
1     Imperial China                                8.090%
2     Russian Empire / USSR                         2.151%
3     Roman civilization                            1.908%
4     Japan (all periods)                           1.611%
5     United States (broad)                         1.578%
6     German Empire (broad)                         0.616%
7     Ancient Egypt                                 0.363%
8     Mongol Empire (broad)                         0.191%


In [19]:
# --- Overlay groups by person-years ---

def count_overlay_py(members, df, results, sample_records, merge_fn, weights_norm, llm_weight_scale):
    """Count person-years for an overlay group using inverse-birth-rate weights."""
    member_ranges = {}
    for name, start, end in members:
        member_ranges.setdefault(name, []).append((start, end))
    
    def matches(polity, year):
        ranges = member_ranges.get(polity)
        if not ranges:
            return False
        return any(
            (start is None or year >= start) and (end is None or year <= end)
            for start, end in ranges
        )
    
    # Deterministic
    det_total = 0
    assigned = df[df['status'] == 'assigned']
    for idx, row in assigned.iterrows():
        polity = merge_fn(row['polity'])
        if matches(polity, row['birth_year']):
            det_total += weights_norm[idx]
    
    # LLM sample
    idx_to_year = {rec[0]: rec[3] for rec in sample_records}
    llm_total = 0
    for orig_idx, polity_name in results.items():
        polity = merge_fn(polity_name)
        year = idx_to_year.get(int(orig_idx))
        if year is not None and matches(polity, year):
            llm_total += weights_norm[int(orig_idx)]
    
    return det_total + llm_total * llm_weight_scale

print(f"{'Overlay group':<35} {'% person-years':>14}")
print('-' * 52)
overlay_py = {}
for group_name, members in sorted(POLITY_OVERLAYS.items()):
    w = count_overlay_py(
        members, df, results, sample_records, merge_polity_name,
        weights_norm, llm_weight_scale
    )
    overlay_py[group_name] = w
    print(f"{group_name:<35} {100*w:>10.3f}%")

Overlay group                       % person-years
----------------------------------------------------
British Empire                           3.719%
Caliphate (mainline)                     0.365%
Dutch Empire                             0.407%
French Empire                            1.249%
Habsburg domains                         0.705%
Mongol Empire (Genghisid)                0.686%
Portuguese Empire                        0.246%
Spanish Empire                           0.988%
